# 日本株における収益性ファクターの再検証：測り方を変えると効くのか？

**仮説**：日本で収益性ファクター（Fama-French の RMW）が効かないと言われるのは、収益性そのものに予測力がないからではなく、
**収益性の測り方が悪い**からではないか。

**ベース論文**：Novy-Marx & Medhat (2025) "Profitability Retrospective: What Have We Learned?" (NBER WP 33601)

論文の主張と、このノートブックで使う定義は次のとおりです。

| 論文での主張 | 内容 |
|---|---|
| 収益性の定義 | **PROF = [売上 − 売上原価 − (販管費 − 研究開発費) − 支払利息] / (自己資本 + 非支配株主持分)** |
| 何が違うか | FF(2015) の RMW は R&D を費用として引いた営業利益を使う。R&D は「将来の売上のための投資」なので、引かないほうが予測力が高い |
| キャッシュベース | 発生主義の調整（アクルーアル控除）を入れても予測力はほとんど上がらない（付録 B） |
| ファクターの作り方 | 規模 × 収益性の 2×3 ソート。米国以外では**累積時価総額上位90%を大型**、収益性の閾値は**大型株内の30/70パーセンタイル**。6月末に年1回リバランス、時価総額加重 |
| 評価方法 | 平均リターン・ファクターモデルのアルファ・**スパニングテスト**（PROF と他の収益性・クオリティ指標を相互に回帰） |

## 日本のデータへの対応付け

日本基準では「営業利益 = 売上総利益 − 販管費」なので、販管費の列が無くても論文の分子を作れます。

| 論文（Compustat） | このノートブック（IDSQE） |
|---|---|
| REVT − COGS（売上総利益） | `GPLS` |
| REVT − COGS − XSGA（R&D控除後の営業利益） | `OPINCM` |
| XRD（研究開発費） | `RD`（欠損は0とみなす。論文と同じ扱い） |
| XINT（支払利息） | `CHAGE`（欠損は0） |
| BE + MIB（自己資本 + 非支配株主持分） | `NTASTTL`（純資産合計。2006年以降は非支配株主持分を含む）→ 欠損なら `EQTY`（資本合計） |
| BE（自己資本：ROE・簿価時価比率用） | `CAPSTTL`（株主資本）→ `EQTY` → `NTASTTL` |
| アクルーアル（Ball et al. 2016） | Δ売上債権 `ACRC` + Δ棚卸資産 `INVENT` − Δ買入債務 `ACPY`（前払費用・前受収益・未払費用は取得していないため省略） |

よって **PROF = (OPINCM + RD − CHAGE) / BE+MIB**、FF型の **RMW_OP = (OPINCM − CHAGE) / BE+MIB** です。
両者の違いは「R&Dを足し戻すかどうか」**だけ**なので、同じユニバース・同じ作り方で比較すれば、測り方の効果だけを取り出せます。

## 分析の流れ

1. 設定（分析開始日など）
2. データの読み込み（上のセルで取得済みならそれを再利用）
3. データの整形（欠損値・業種表の重複・決算データのキー）
4. 年次の決算パネルと収益性指標の計算
5. TOPIXユニバース・証券コード対応・月次市場データの作成
6. 6月末時点のポートフォリオ組成用テーブル（決算の公表日を考慮）
7. 診断：銘柄数・R&Dのカバレッジ・PROFとRMWがどれくらい違うか
8. 2×3ソートによるファクター構築（比較用の指標・自前のSMB/HML/CMA/UMDも作る）
9. ファクターの成績
10. 5分位ポートフォリオ（単調性の確認）
11. ファクターモデルに対するアルファ
12. スパニングテスト（PROF vs RMW）← **本題**
13. 指標間の総当たりスパニングテスト（論文の Fig. 2 に相当）
14. 最大シャープレシオの比較
15. Fama-MacBeth 回帰（論文の Table B3 に相当）
16. 期間別の頑健性
17. 結果の読み方

> 前提：このノートブックのセルは、データ取得関数（`get_pl_data` など）を定義したセルの**下に追加して上から順に実行**する想定です。
> `statsmodels` などの追加ライブラリは使わず、pandas / numpy / matplotlib だけで動きます。

## 1. 設定

分析期間などはここで変更します。

- `ANALYSIS_START`：分析開始月（この月以降の月次リターンで評価）。`data/Japan_5_Factors.csv` は 1990-07 から始まるので、それより前を指定するとスパニングテストの標本は 1990-07 からになります。
- `ANALYSIS_END`：終了月。`None` ならデータの最終月まで。
- `SUBPERIODS`：期間別分析の区切り。論文は 2007年以降（クオンツ・ショック後）と 2013年の Novy-Marx 論文公表後に注目しています。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import japanize_matplotlib  # noqa: F401  （入っていれば図の日本語が表示される）
except ImportError:
    pass

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

# ===== 分析設定 =====
ANALYSIS_START = "1995-07"        # 分析開始月（yyyy-mm）
ANALYSIS_END = None               # 終了月（yyyy-mm）。None ならデータの最終月まで
FF_CSV_PATH = "data/Japan_5_Factors.csv"

REBALANCE_MONTH = 6               # 毎年この月末にポートフォリオを組み直す（論文は6月末）
LARGE_CAP_CUM_SHARE = 0.90        # 時価総額の大きい順に累積90%までを大型、残りを小型
SIGNAL_BREAKS = (0.30, 0.70)      # 2×3ソートの閾値（大型株内のパーセンタイル）
N_QUANTILES = 5                   # 単調性確認用の分位数（閾値は大型株内で計算）

EXCLUDE_FINANCIALS = True         # 銀行・証券・保険・その他金融を除く（売上総利益が定義できないため）
CTYPE_PRIORITY = ["2", "1", "3"]  # 同じ決算期に複数の会計基準がある場合の優先順（2:日本 1:SEC 3:IFRS）
FALLBACK_LAG_MONTHS = 2           # 決算発表日・短信入手日がどちらも欠損なら、決算日 + 2か月で公表とみなす
MAX_STALENESS_MONTHS = 18         # 組成時点でこれより古い決算は使わない
ACCOUNTING_UNIT = 1_000_000       # 財務データの単位（百万円）。時価総額（円）との比率計算に使う

NW_LAGS = 6                       # 時系列回帰の Newey-West ラグ
INCLUDE_UMD = True                # スパニングテストの説明変数に自前のモメンタム(UMD)を入れるか（論文は入れている）
LOCAL_RF_MONTHLY = 0.0            # 自前の市場ファクター用の円建て無リスク金利（月次・小数）。入手できれば Series でも可
FMB_TRIM = (0.01, 0.99)           # Fama-MacBeth 回帰の説明変数を毎月この分位で切り落とす
FMB_MIN_OBS = 50                  # Fama-MacBeth で1か月に必要な最低銘柄数

SUBPERIODS = [
    (None, "2006-12"),
    ("2007-01", None),
    ("2014-01", None),            # Novy-Marx (2013) 公表後
]

SAVE_OUTPUTS = True               # ファクターリターンなどを output/ に CSV で保存
OUTPUT_DIR = Path("output")

# 東証33業種コード（業種表の IND_NAME は JOIN 条件の都合で正しく対応していないため、コードから付け直す）
IND33_NAMES = {
    "0050": "水産・農林業", "1050": "鉱業", "2050": "建設業", "3050": "食料品", "3100": "繊維製品",
    "3150": "パルプ・紙", "3200": "化学", "3250": "医薬品", "3300": "石油・石炭製品", "3350": "ゴム製品",
    "3400": "ガラス・土石製品", "3450": "鉄鋼", "3500": "非鉄金属", "3550": "金属製品", "3600": "機械",
    "3650": "電気機器", "3700": "輸送用機器", "3750": "精密機器", "3800": "その他製品", "4050": "電気・ガス業",
    "5050": "陸運業", "5100": "海運業", "5150": "空運業", "5200": "倉庫・運輸関連業", "5250": "情報・通信業",
    "6050": "卸売業", "6100": "小売業", "7050": "銀行業", "7100": "証券、商品先物取引業", "7150": "保険業",
    "7200": "その他金融業", "8050": "不動産業", "9050": "サービス業",
}
FIN_CODES = {"7050", "7100", "7150", "7200"}

def show(df, fmt="{:.3f}", cmap=None, vmin=None, vmax=None, subset=None, axis=0):
    """書式付きで表を表示（jinja2 が無い環境では丸めて表示）"""
    try:
        sty = df.style.format(fmt, na_rep="")
        if cmap:
            sty = sty.background_gradient(cmap=cmap, vmin=vmin, vmax=vmax, subset=subset, axis=axis)
        display(sty)
    except (AttributeError, ImportError):
        display(df.round(3))


START_P = pd.Period(ANALYSIS_START, freq="M")
END_P = pd.Period(ANALYSIS_END, freq="M") if ANALYSIS_END else None
print(f"分析期間：{START_P} 〜 {END_P or 'データ最終月'}")

## 2. データの読み込み

上のセルで `取得データ` 辞書を作っていればそれを再利用し、無ければ各取得関数を呼びます（全件で40秒程度）。

In [ ]:
def _load(name, func):
    store = globals().get("取得データ")
    if isinstance(store, dict) and name in store:
        print(f"{name}: 取得済みデータを再利用（{len(store[name]):,} 行）")
        return store[name].copy()
    df = func()
    print(f"{name}: {func.__name__}() で取得（{len(df):,} 行）")
    return df


raw = {
    name: _load(name, func)
    for name, func in [
        ("PL", get_pl_data),
        ("BS", get_bs_data),
        ("決算詳細", get_accounting_details),
        ("証券コード対応表", get_nri_to_ticker_table),
        ("業種", get_industry),
        ("TOPIX構成銘柄", get_topix_data),
        ("時価総額", get_market_cap_data),
        ("月次リターン", get_return_data),
    ]
}

## 3. データの整形

取得結果の確認で分かった問題をここで処理します。

- **欠損値**：IDSQE 系は欠損が `-999999998` なので NaN に置き換える。
- **文字列のキー**：`NRI_CODE`・`CTYPE`・`SETTLEMENT_FLG` などを前後の空白を除いた文字列にそろえる。
- **業種表の重複**：JOIN 条件が業種区分だけなので行が膨らみ、業種名もずれている。業種コード（`ITEM_DATA`）で重複を除き、名称はコードから付け直す。
- **決算データのキー**：PL・BS には `SETTLEMENT_FLG`（本決算／中間／四半期）が無い。`(NRI_CODE, FTERM, CTYPE)` が重複している場合は、本決算と中間決算が混ざっているので、`SETTLEMENT_FLG` 付きで取り直してから結合する（重複が無ければそのまま使う）。

In [ ]:
MISSING_THRESHOLD = -999_999_990
STR_COLS = {"NRI_CODE", "CTYPE", "SETTLEMENT_FLG", "ITEM_DATA", "IND_NAME", "ITEM"}


def clean_idsqe(df):
    """IDSQE 系の欠損値（-999999998）を NaN にし、キー列を文字列にそろえる"""
    df = df.copy()
    for c in df.columns:
        if c in STR_COLS:
            df[c] = df[c].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        elif pd.api.types.is_numeric_dtype(df[c]):
            s = df[c].astype(float)
            df[c] = s.where(s > MISSING_THRESHOLD)
    return df


KEYS = ["NRI_CODE", "FTERM", "CTYPE"]
pl = clean_idsqe(raw["PL"])
bs = clean_idsqe(raw["BS"])
at = clean_idsqe(raw["決算詳細"])

dups = {n: int(d.duplicated(KEYS).sum()) for n, d in [("PL", pl), ("BS", bs), ("決算詳細", at)]}
print("(NRI_CODE, FTERM, CTYPE) の重複行数：", dups)

if "SETTLEMENT_FLG" in pl.columns and "SETTLEMENT_FLG" in bs.columns:
    MERGE_KEYS = KEYS + ["SETTLEMENT_FLG"]
elif dups["PL"] or dups["BS"]:
    print("→ PL/BS に本決算以外の行が混ざっているため、SETTLEMENT_FLG 付きで取り直します")

    def _refetch(table, cols):
        col_sql = ",".join(f"[{c}]" for c in KEYS + ["SETTLEMENT_FLG"] + cols)
        return clean_idsqe(fetch_data(f"SELECT {col_sql} FROM [IDSQE].[dbo].[{table}]", "idsqe"))

    pl = _refetch("NRIFIN_CON_PL4", [c for c in pl.columns if c not in KEYS])
    bs = _refetch("NRIFIN_CON_BS4", [c for c in bs.columns if c not in KEYS])
    MERGE_KEYS = KEYS + ["SETTLEMENT_FLG"]
else:
    MERGE_KEYS = KEYS

acct_raw = (
    at.drop_duplicates(MERGE_KEYS if set(MERGE_KEYS) <= set(at.columns) else KEYS, keep="last")
    .merge(pl.drop_duplicates(MERGE_KEYS, keep="last"), on=MERGE_KEYS, how="left")
    .merge(bs.drop_duplicates(MERGE_KEYS, keep="last"), on=MERGE_KEYS, how="left")
)
print(f"決算データ結合後：{len(acct_raw):,} 行（結合キー {MERGE_KEYS}）")

# 業種：重複を除き、名称をコードから付け直す
ind = (
    clean_idsqe(raw["業種"])
    .drop(columns=["IND_NAME"], errors="ignore")
    .drop_duplicates(["NRI_CODE", "FROM_DATE", "TO_DATE", "ITEM_DATA"])
    .rename(columns={"ITEM_DATA": "IND33"})
)
ind["FROM_DATE"] = ind["FROM_DATE"].fillna(0)
ind["TO_DATE"] = ind["TO_DATE"].fillna(99_999_999)
ind["IND_NAME"] = ind["IND33"].map(IND33_NAMES)
print(f"業種：{len(raw['業種']):,} 行 → 重複除去後 {len(ind):,} 行（名称が付かないコード：{ind['IND_NAME'].isna().sum():,} 行）")

# TOPIX 構成銘柄・証券コード対応表
topix = clean_idsqe(raw["TOPIX構成銘柄"])[["NRI_CODE", "FROM_DATE", "TO_DATE"]]
topix["FROM_DATE"] = topix["FROM_DATE"].fillna(0)
topix["TO_DATE"] = topix["TO_DATE"].fillna(99_999_999)

code_map = clean_idsqe(raw["証券コード対応表"]).rename(columns={"ITEM_DATA": "security_code"})
code_map["security_code"] = code_map["security_code"].str.upper()
code_map["FROM_DATE"] = code_map["FROM_DATE"].fillna(0)
code_map["TO_DATE"] = code_map["TO_DATE"].fillna(99_999_999)

# 月次リターン・時価総額（銘柄×月の重複は最後の行を残す）
ret_m = raw["月次リターン"][["security_code", "month", "ret"]].copy()
ret_m["security_code"] = ret_m["security_code"].astype(str).str.strip().str.upper()
ret_m = ret_m.dropna(subset=["ret"])
n_dup_ret = int(ret_m.duplicated(["security_code", "month"]).sum())
ret_m = ret_m.drop_duplicates(["security_code", "month"], keep="last")

mcap_m = raw["時価総額"][["security_code", "month", "market_cap"]].copy()
mcap_m["security_code"] = mcap_m["security_code"].astype(str).str.strip().str.upper()
mcap_m = mcap_m[mcap_m["market_cap"] > 0]
n_dup_cap = int(mcap_m.duplicated(["security_code", "month"]).sum())
mcap_m = mcap_m.drop_duplicates(["security_code", "month"], keep="last")
print(f"月次リターンの重複：{n_dup_ret:,} 行、時価総額の重複：{n_dup_cap:,} 行（いずれも最後の行を残した）")

## 4. 年次の決算パネルと収益性指標

**使う決算**：本決算（`SETTLEMENT_FLG = 1`）かつ決算月数12か月の連結決算。同じ決算期に複数の会計基準があれば `CTYPE_PRIORITY` の順で1つ選びます。

**公表日**：決算発表日（`ANNOUNCE`）と短信入手日（`RCVD`）の早いほう。どちらも欠損なら決算日 + `FALLBACK_LAG_MONTHS` か月の月末。
組成時点でまだ公表されていない決算は使わないので、先読みバイアスを防げます。

**前期の値**：アクルーアルと総資産成長率のために、9〜15か月前の決算を前期として使います。

比較する収益性指標（論文の Table 1・Table B1 に対応）：

| 名前 | 定義 | 出典 | ねらい |
|---|---|---|---|
| **PROF** | (営業利益 + R&D − 支払利息) / (BE+MIB) | Novy-Marx & Medhat (2025) | 論文の提案指標 |
| RMW_OP | (営業利益 − 支払利息) / (BE+MIB) | Fama & French (2015) | R&Dを引く従来型。PROFとの差は R&D の扱いだけ |
| COP | (営業利益 − 支払利息 − アクルーアル) / (BE+MIB) | Fama & French (2018) | キャッシュベース |
| COP_RD | (営業利益 + R&D − 支払利息 − アクルーアル) / (BE+MIB) | 論文 付録B | キャッシュベース + R&D足し戻し |
| GPA | 売上総利益 / 総資産 | Novy-Marx (2013) | 粗利益 |
| OPRD_A | (営業利益 + R&D) / 総資産 | Ball et al. (2015) | 資産ベースの営業利益（R&D足し戻し） |
| ROE | 当期純利益 / BE | GMO (2004) など | 実務でよく使う「クオリティ」 |
| ROA | 当期純利益 / 総資産 | — | 同上 |

加えて、自前の HML・CMA を作るために簿価時価比率（BM）と総資産成長率（INV）も計算します。
また、外部の FF5 ファクター（`FF_RMW`）も比較対象に入れます。

In [ ]:
acct = acct_raw.copy()
if "SETTLEMENT_FLG" in acct.columns:
    acct = acct[acct["SETTLEMENT_FLG"] == "1"]
acct = acct[acct["NMONTH"] == 12]

# 会計基準の優先順で1決算期1行にする
rank = {c: i for i, c in enumerate(CTYPE_PRIORITY)}
acct["_rank"] = acct["CTYPE"].map(rank).fillna(99)
acct = acct.sort_values(["NRI_CODE", "FTERM", "_rank"]).drop_duplicates(["NRI_CODE", "FTERM"], keep="first")


def int_to_date(s):
    """yyyymmdd の数値を Timestamp に（欠損・不正値は NaT）"""
    return pd.to_datetime(s.astype("Int64").astype(str), format="%Y%m%d", errors="coerce").astype("datetime64[ns]")


fterm_end = (
    pd.to_datetime(acct["FTERM"].astype("Int64").astype(str), format="%Y%m", errors="coerce") + pd.offsets.MonthEnd(0)
).astype("datetime64[ns]")
acct["fdate"] = int_to_date(acct["FDATE"]).fillna(fterm_end)

ann = int_to_date(acct["ANNOUNCE"])
rcv = int_to_date(acct["RCVD"])
ann = ann.where(ann >= acct["fdate"])
rcv = rcv.where(rcv >= acct["fdate"])
acct["avail_date"] = pd.concat([ann, rcv], axis=1).min(axis=1)
fallback = (acct["fdate"] + pd.offsets.MonthEnd(FALLBACK_LAG_MONTHS)).astype("datetime64[ns]")
acct["avail_src"] = np.where(ann.notna() | rcv.notna(), "発表日/入手日", "決算日+ラグ")
acct["avail_date"] = acct["avail_date"].fillna(fallback).astype("datetime64[ns]")
acct = acct.dropna(subset=["fdate"])

# 前期の値（9〜15か月前の決算のみ有効）
acct = acct.sort_values(["NRI_CODE", "fdate"])
g = acct.groupby("NRI_CODE", sort=False)
gap = (acct["fdate"] - g["fdate"].shift(1)).dt.days / 30.4
has_prev = gap.between(9 * 0.97, 15 * 1.03)
for c in ["TOTALBS", "ACRC", "INVENT", "ACPY"]:
    acct[f"{c}_lag"] = g[c].shift(1).where(has_prev)
acct["has_prev"] = has_prev

# 分母
acct["BEMIB"] = acct["NTASTTL"].fillna(acct["EQTY"])                       # 自己資本 + 非支配株主持分
acct["BE"] = acct["CAPSTTL"].fillna(acct["EQTY"]).fillna(acct["NTASTTL"])  # 自己資本
bemib = acct["BEMIB"].where(acct["BEMIB"] > 0)
be = acct["BE"].where(acct["BE"] > 0)
ta = acct["TOTALBS"].where(acct["TOTALBS"] > 0)

# 分子の部品（論文どおり、R&D・支払利息の欠損は0）
rd0 = acct["RD"].fillna(0).clip(lower=0)
int0 = acct["CHAGE"].fillna(0)
d_acrc = (acct["ACRC"] - acct["ACRC_lag"]).fillna(0)
d_inv = (acct["INVENT"] - acct["INVENT_lag"]).fillna(0)
d_acpy = (acct["ACPY"] - acct["ACPY_lag"]).fillna(0)
acct["ACC"] = (d_acrc + d_inv - d_acpy).where(acct["has_prev"])            # Ball et al. (2016) の貸借対照表アプローチ（簡略版）

acct["PROF"] = (acct["OPINCM"] + rd0 - int0) / bemib
acct["RMW_OP"] = (acct["OPINCM"] - int0) / bemib
acct["COP"] = (acct["OPINCM"] - int0 - acct["ACC"]) / bemib
acct["COP_RD"] = (acct["OPINCM"] + rd0 - int0 - acct["ACC"]) / bemib
acct["GPA"] = acct["GPLS"] / ta
acct["OPRD_A"] = (acct["OPINCM"] + rd0) / ta
acct["ROE"] = acct["NETINCM"] / be
acct["ROA"] = acct["NETINCM"] / ta
acct["INV"] = acct["TOTALBS"] / acct["TOTALBS_lag"].where(acct["TOTALBS_lag"] > 0) - 1
acct["RD_RATIO"] = rd0 / bemib                                             # 足し戻した R&D の大きさ

PROFIT_SIGNALS = ["PROF", "RMW_OP", "COP", "COP_RD", "GPA", "OPRD_A", "ROE", "ROA"]
SIGNAL_LABELS = {
    "PROF": "PROF（論文：営業利益+R&D−利息 / BE+MIB）",
    "RMW_OP": "RMW_OP（FF2015：営業利益−利息 / BE+MIB）",
    "COP": "COP（FF2018：キャッシュベース）",
    "COP_RD": "COP_RD（キャッシュベース + R&D足し戻し）",
    "GPA": "GPA（Novy-Marx 2013：粗利益/総資産）",
    "OPRD_A": "OPRD_A（Ball et al. 2015：(営業利益+R&D)/総資産）",
    "ROE": "ROE（純利益/株主資本）",
    "ROA": "ROA（純利益/総資産）",
    "FF_RMW": "FF_RMW（外部CSVのRMW）",
}

print(f"年次決算パネル：{len(acct):,} 行・{acct['NRI_CODE'].nunique():,} 社")
print("公表日の出所：", acct["avail_src"].value_counts().to_dict())
display(acct[PROFIT_SIGNALS + ["INV", "RD_RATIO"]].describe(percentiles=[0.01, 0.5, 0.99]).T)

## 5. TOPIXユニバース・証券コード対応・月次市場データ

- **TOPIXユニバース**：各月末に `FROM_DATE ≤ 月末日 ≤ TO_DATE` を満たす銘柄（`TO_DATE = 99999999` は現在も構成銘柄）。
- **証券コード**：IDSQE 系の `NRI_CODE` と RISK_MODELS 系の `security_code` を、その月末時点で有効な対応表で結び付ける。
- **業種**：その月末時点で有効な東証33業種。
- **月次パネル**：リターンに、前月末の時価総額（加重用）・過去12〜2か月リターン（モメンタム用）・前月リターンを付ける。

In [ ]:
def month_end_int(p):
    return int(p.to_timestamp(how="end").strftime("%Y%m%d"))


def build_monthly_universe(months):
    out = []
    for m in months:
        d = month_end_int(m)
        mem = topix.loc[(topix["FROM_DATE"] <= d) & (topix["TO_DATE"] >= d), ["NRI_CODE"]].drop_duplicates()
        cm = (
            code_map.loc[(code_map["FROM_DATE"] <= d) & (code_map["TO_DATE"] >= d)]
            .sort_values("FROM_DATE")
            .drop_duplicates("NRI_CODE", keep="last")[["NRI_CODE", "security_code"]]
        )
        im = (
            ind.loc[(ind["FROM_DATE"] <= d) & (ind["TO_DATE"] >= d)]
            .sort_values("FROM_DATE")
            .drop_duplicates("NRI_CODE", keep="last")[["NRI_CODE", "IND33"]]
        )
        u = mem.merge(cm, on="NRI_CODE", how="left").merge(im, on="NRI_CODE", how="left")
        u["month"] = m
        out.append(u)
    u = pd.concat(out, ignore_index=True)
    u["month"] = u["month"].astype("period[M]")
    return u


R = REBALANCE_MONTH
last_month = ret_m["month"].max()
if END_P is not None:
    last_month = min(last_month, END_P)
first_form_year = START_P.year if START_P.month > R else START_P.year - 1
last_form_year = last_month.year if last_month.month > R else last_month.year - 1
first_form_month = pd.Period(year=first_form_year, month=R, day=1, freq="M")
univ_months = pd.period_range(first_form_month - 1, last_month, freq="M")

univ_m = build_monthly_universe(univ_months)
univ_m["is_fin"] = univ_m["IND33"].isin(FIN_CODES)
print(f"TOPIXユニバース：{univ_months[0]}〜{univ_months[-1]}、平均 {univ_m.groupby('month').size().mean():,.0f} 銘柄/月")
print(f"証券コードが対応しない行の割合：{univ_m['security_code'].isna().mean():.2%}、業種が無い行の割合：{univ_m['IND33'].isna().mean():.2%}")

# 月次パネル：リターン + 前月末時価総額 + モメンタム
me_lag = mcap_m.rename(columns={"market_cap": "me_lag"}).assign(month=lambda d: d["month"] + 1)
R_wide = ret_m.pivot(index="month", columns="security_code", values="ret")
R_wide = R_wide.reindex(pd.period_range(R_wide.index.min(), R_wide.index.max(), freq="M"))
log_r = np.log1p(R_wide.clip(lower=-0.9999))
mom_wide = np.expm1(log_r.shift(2).rolling(11, min_periods=8).sum())   # t-12〜t-2 の累積リターン
r1_wide = R_wide.shift(1)                                              # t-1 のリターン


def _long(w, name):
    s = w.stack()
    s.index.names = ["month", "security_code"]
    return s.dropna().rename(name).reset_index()


panel = (
    ret_m.merge(me_lag, on=["security_code", "month"], how="left")
    .merge(_long(mom_wide, "r12_2"), on=["security_code", "month"], how="left")
    .merge(_long(r1_wide, "r1_0"), on=["security_code", "month"], how="left")
)
panel = panel[(panel["month"] > first_form_month) & (panel["month"] <= last_month)]
print(f"月次パネル：{len(panel):,} 行")

## 6. ポートフォリオ組成用テーブル（毎年6月末）

毎年6月末（`REBALANCE_MONTH`）に、その時点の TOPIX 構成銘柄（金融を除く）について次を用意します。

- **6月末の時価総額**：規模の分類（累積時価総額90%までを大型）に使う。
- **前年12月末の時価総額**：簿価時価比率の分母（Fama-French と同じ）。
- **決算データ**：6月末までに公表済みの最新の本決算（`pd.merge_asof` で公表日基準に結合）。`MAX_STALENESS_MONTHS` より古い決算は使わない。

組成したポートフォリオは7月から翌年6月まで保有し、毎月、前月末時価総額で加重したリターンを計算します。

In [ ]:
form_months = pd.PeriodIndex(
    [pd.Period(year=y, month=R, day=1, freq="M") for y in range(first_form_year, last_form_year + 1)]
)
base = univ_m[univ_m["month"].isin(form_months)].dropna(subset=["security_code"]).copy()
if EXCLUDE_FINANCIALS:
    base = base[~base["is_fin"]]

base = base.merge(mcap_m.rename(columns={"market_cap": "ME"}), on=["security_code", "month"], how="left")
me_dec = mcap_m.rename(columns={"market_cap": "ME_DEC"}).assign(month=lambda d: d["month"] + R)  # 前年12月末
base = base.merge(me_dec, on=["security_code", "month"], how="left")
base["form_date"] = (base["month"].dt.to_timestamp(how="end").dt.normalize()).astype("datetime64[ns]")

acct_cols = ["NRI_CODE", "avail_date", "fdate", "FTERM", "CTYPE", "BE", "BEMIB", "TOTALBS", "RD",
             "OPINCM", "GPLS", "CHAGE"] + PROFIT_SIGNALS + ["INV", "RD_RATIO"]
form = pd.merge_asof(
    base.sort_values("form_date"),
    acct[acct_cols].dropna(subset=["avail_date"]).sort_values("avail_date"),
    left_on="form_date", right_on="avail_date", by="NRI_CODE", direction="backward",
)
stale = form["fdate"] < form["form_date"] - pd.DateOffset(months=MAX_STALENESS_MONTHS)
form.loc[stale, acct_cols[1:]] = np.nan

form["BM"] = (form["BE"] * ACCOUNTING_UNIT / form["ME_DEC"]).where(form["BE"] > 0)
form["NEG_INV"] = -form["INV"]           # CMA は「総資産成長率が低い」ほうを買うので符号を反転
form["form_year"] = form["month"].dt.year

# 同じ組成月に同じ証券コードが2社に対応してしまう場合は1社に絞る
n_dup_form = int(form.duplicated(["month", "security_code"]).sum())
form = form.sort_values(["month", "security_code", "ME"]).drop_duplicates(["month", "security_code"], keep="last")

# 規模：時価総額の大きい順に累積90%までを大型（B）、残りを小型（S）
def assign_size(df, me_col, by="month"):
    d = df.sort_values([by, me_col], ascending=[True, False])
    tot = d.groupby(by)[me_col].transform("sum")
    cum_prev = d.groupby(by)[me_col].cumsum() - d[me_col]
    return pd.Series(np.where(cum_prev / tot < LARGE_CAP_CUM_SHARE, "B", "S"), index=d.index).reindex(df.index)


eligible = (form["ME"] > 0) & form["fdate"].notna()
form["SIZE"] = pd.Series(pd.NA, index=form.index, dtype="object")
form.loc[eligible, "SIZE"] = assign_size(form[eligible], "ME")

print(f"組成テーブル：{len(form):,} 行（重複除去 {n_dup_form} 行）")
cov = form.groupby("form_year").agg(
    TOPIX非金融=("security_code", "size"),
    時価総額あり=("ME", lambda s: s.notna().sum()),
    決算あり=("fdate", lambda s: s.notna().sum()),
    PROFあり=("PROF", lambda s: s.notna().sum()),
    大型=("SIZE", lambda s: (s == "B").sum()),
    小型=("SIZE", lambda s: (s == "S").sum()),
)
display(cov)

## 7. 診断：PROF と RMW はどれくらい違うのか

PROF と RMW_OP の違いは R&D を足し戻すかどうかだけです。日本企業で R&D を開示している企業が少なかったり、
金額が小さかったりすると、2つの指標はほとんど同じになり、「測り方を変えても結果が変わらない」のは当然になります。
そこで次を確認します。

- R&D を開示している（`RD > 0`）銘柄の割合と、足し戻す R&D の大きさ（R&D / BE+MIB）
- PROF と RMW_OP の順位相関（毎年の断面での Spearman 相関）
- 2×3 ソートで高・中・低の区分が変わる銘柄の割合

> 注意：日本基準では R&D の一部が製造原価（売上原価）に含まれることがあります。`RD` が注記の研究開発費総額なら、
> 営業利益に足し戻すと売上原価側の R&D も戻すことになり、論文（販管費からだけ引き戻す）より少し広い調整になります。

In [ ]:
def spearman_by(df, a, b, by="form_year"):
    d = df[[by, a, b]].dropna()
    return pd.Series({k: x[a].rank().corr(x[b].rank()) for k, x in d.groupby(by)})


diag = form[form["SIZE"].notna()].copy()
diag["RD開示"] = diag["RD"] > 0
w = diag["ME"]
rd_diag = pd.DataFrame({
    "R&D開示銘柄の割合": diag.groupby("form_year")["RD開示"].mean(),
    "R&D開示銘柄の割合(時価総額加重)": (diag["RD開示"] * w).groupby(diag["form_year"]).sum() / w.groupby(diag["form_year"]).sum(),
    "R&D/BE+MIB 平均(開示企業)": diag[diag["RD開示"]].groupby("form_year")["RD_RATIO"].mean(),
    "PROFとRMW_OPの順位相関": spearman_by(diag, "PROF", "RMW_OP"),
    "PROFとGPAの順位相関": spearman_by(diag, "PROF", "GPA"),
})


def bucket_30_70(df, col, by="form_year"):
    big = df[df["SIZE"] == "B"]
    lo = df[by].map(big.groupby(by)[col].quantile(SIGNAL_BREAKS[0]))
    hi = df[by].map(big.groupby(by)[col].quantile(SIGNAL_BREAKS[1]))
    return np.select([df[col] <= lo, df[col] >= hi], ["L", "H"], "M")


both = diag.dropna(subset=["PROF", "RMW_OP"]).copy()
both["b_prof"] = bucket_30_70(both, "PROF")
both["b_rmw"] = bucket_30_70(both, "RMW_OP")
rd_diag["高中低の区分が変わる銘柄の割合"] = (both["b_prof"] != both["b_rmw"]).groupby(both["form_year"]).mean()
show(rd_diag)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
rd_diag[["R&D開示銘柄の割合", "R&D開示銘柄の割合(時価総額加重)"]].plot(ax=ax[0], marker="o")
ax[0].set_title("Share of firms reporting R&D")
ax[0].set_ylim(0, 1)
rd_diag[["PROFとRMW_OPの順位相関", "高中低の区分が変わる銘柄の割合"]].plot(ax=ax[1], marker="o")
ax[1].set_title("PROF vs RMW_OP: rank corr. / bucket changes")
plt.tight_layout()
plt.show()

# 業種別：R&D を足し戻すと収益性がどれだけ上がるか（直近の組成年）
latest = diag[diag["form_year"] == diag["form_year"].max()]
by_ind = latest.groupby(latest["IND33"].map(IND33_NAMES)).agg(
    銘柄数=("security_code", "size"),
    RD開示割合=("RD開示", "mean"),
    PROF中央値=("PROF", "median"),
    RMW_OP中央値=("RMW_OP", "median"),
    RD比率中央値=("RD_RATIO", "median"),
).sort_values("RD比率中央値", ascending=False)
print(f"業種別（{diag['form_year'].max()}年6月組成）")
show(by_ind, {"銘柄数": "{:.0f}", "RD開示割合": "{:.2f}", "PROF中央値": "{:.3f}", "RMW_OP中央値": "{:.3f}", "RD比率中央値": "{:.3f}"})

# 収益性指標どうしの順位相関（毎年の断面相関の平均）
corr_list = []
for y, d in diag.groupby("form_year"):
    corr_list.append(d[PROFIT_SIGNALS + ["BM", "INV"]].rank().corr())
avg_corr = sum(corr_list) / len(corr_list)
print("指標間の順位相関（毎年の断面の平均）")
show(avg_corr, "{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1)

## 8. ファクターの構築（2×3ソート）

論文（付録 A.3）の米国以外の作り方に合わせます。

1. 6月末に、時価総額上位から累積90%までを**大型（B）**、残りを**小型（S）**に分ける。
2. 指標の30・70パーセンタイルを**大型株の中だけ**で計算し、全銘柄を低（L）・中（M）・高（H）に分ける。
3. 6つのポートフォリオ（S/L, S/M, S/H, B/L, B/M, B/H）の時価総額加重リターンを7月〜翌年6月まで毎月計算する。
4. **ファクター = (S/H + B/H)/2 − (S/L + B/L)/2**

同じ手順で、比較対象の収益性指標すべてと、自前の HML（BM）・CMA（総資産成長率の低さ）を作ります。
SMB は Fama-French と同様に、BM・RMW_OP・INV の3つのソートの「小型3つの平均 − 大型3つの平均」を平均します。

**モメンタム（UMD）**：外部CSVには無いので自前で作ります。毎月末の TOPIX 構成銘柄（金融を含む）を前月末時価総額で大型・小型に分け、
過去12〜2か月リターンで同じ 2×3 ソートを行い、毎月リバランスします。

**自前の市場ファクター（MKT_L）**：TOPIXユニバースの時価総額加重リターン − 円金利（`LOCAL_RF_MONTHLY`）。
外部CSVの FF ファクターはドル建て（Ken French のデータ）なので、円建ての自前ファクターでも頑健性を確認します。

In [ ]:
def assign_bucket(df, col, by="month", breaks=SIGNAL_BREAKS, size_col="SIZE"):
    """大型株内のパーセンタイルで L/M/H に分ける"""
    ok = df[col].notna() & df[size_col].notna()
    big = df[ok & (df[size_col] == "B")]
    bp = big.groupby(by)[col].quantile(list(breaks)).unstack()
    lo = df[by].map(bp[breaks[0]])
    hi = df[by].map(bp[breaks[1]])
    out = pd.Series(pd.NA, index=df.index, dtype="object")
    out[ok & (df[col] <= lo)] = "L"
    out[ok & (df[col] >= hi)] = "H"
    out[ok & (df[col] > lo) & (df[col] < hi)] = "M"
    return out


def assign_quantile(df, col, q=N_QUANTILES, by="month", size_col="SIZE"):
    """大型株内の分位点で全銘柄を 1..q に分ける"""
    ok = df[col].notna() & df[size_col].notna()
    big = df[ok & (df[size_col] == "B")]
    probs = [i / q for i in range(1, q)]
    bp = big.groupby(by)[col].quantile(probs).unstack()
    rank = pd.Series(1.0, index=df.index)
    for p in probs:
        rank += (df[col] > df[by].map(bp[p])).astype(float)
    return rank.where(ok)


def group_vw(h, group_col):
    d = h.loc[h[group_col].notna() & h["ret"].notna() & (h["me_lag"] > 0), ["month", group_col, "ret", "me_lag"]]
    d = d.assign(wr=d["ret"] * d["me_lag"])
    gg = d.groupby(["month", group_col])
    return (gg["wr"].sum() / gg["me_lag"].sum()).unstack(group_col), gg.size().unstack(group_col)


def factor_from_2x3(h, bucket_col, size_col="SIZE"):
    lab = h[size_col].astype("object") + "/" + h[bucket_col].astype("object")
    lab = lab.where(h[size_col].notna() & h[bucket_col].notna())
    vw, cnt = group_vw(h.assign(_g=lab), "_g")
    vw = vw.reindex(columns=["S/L", "S/M", "S/H", "B/L", "B/M", "B/H"])
    fac = 0.5 * (vw["S/H"] + vw["B/H"]) - 0.5 * (vw["S/L"] + vw["B/L"])
    smb = vw[["S/L", "S/M", "S/H"]].mean(axis=1, skipna=False) - vw[["B/L", "B/M", "B/H"]].mean(axis=1, skipna=False)
    return fac, smb, vw, cnt


SORT_SIGNALS = PROFIT_SIGNALS + ["BM", "NEG_INV"]
for s in SORT_SIGNALS:
    form[f"{s}_b"] = assign_bucket(form, s)
    form[f"{s}_q"] = assign_quantile(form, s)

# 保有期間（7月〜翌年6月）に展開
hold = panel.copy()
hold["form_year"] = np.where(hold["month"].dt.month > R, hold["month"].dt.year, hold["month"].dt.year - 1)
keep_cols = ["security_code", "form_year", "NRI_CODE", "SIZE", "IND33"] + SORT_SIGNALS + [f"{s}_b" for s in SORT_SIGNALS] + [f"{s}_q" for s in SORT_SIGNALS] + ["INV"]
hold = hold.merge(form[keep_cols], on=["security_code", "form_year"], how="inner")
print(f"保有パネル：{len(hold):,} 行")

factors = {}
six_port = {}
port_counts = {}
smb_parts = []
for s in SORT_SIGNALS:
    fac, smb, vw, cnt = factor_from_2x3(hold, f"{s}_b")
    factors[s] = fac
    six_port[s] = vw
    port_counts[s] = cnt
    if s in ("BM", "RMW_OP", "NEG_INV"):
        smb_parts.append(smb)

factors = pd.DataFrame(factors).rename(columns={"BM": "HML_L", "NEG_INV": "CMA_L"})
factors["SMB_L"] = pd.concat(smb_parts, axis=1).mean(axis=1)

# 自前のモメンタム（毎月リバランス）と市場ファクター
u_prev = univ_m[["month", "security_code"]].dropna().drop_duplicates().assign(month=lambda d: d["month"] + 1)
um = panel.merge(u_prev, on=["security_code", "month"], how="inner")
um = um[um["me_lag"] > 0].copy()
um["SIZE"] = assign_size(um, "me_lag")
um["MOM_b"] = assign_bucket(um, "r12_2")
umd, _, _, _ = factor_from_2x3(um, "MOM_b")
factors["UMD"] = umd
mkt = um.assign(wr=um["ret"] * um["me_lag"]).groupby("month")
factors["MKT_L"] = mkt["wr"].sum() / mkt["me_lag"].sum() - LOCAL_RF_MONTHLY

# 外部の FF5（%表示・ドル建て）
ff = pd.read_csv(FF_CSV_PATH)
ff.columns = [c.strip() for c in ff.columns]
ff["Date"] = pd.to_numeric(ff["Date"], errors="coerce")
ff = ff[ff["Date"].between(190001, 210012)].copy()
ff["month"] = pd.PeriodIndex(ff["Date"].astype(int).astype(str), freq="M")
ff = ff.set_index("month").drop(columns="Date").apply(pd.to_numeric, errors="coerce") / 100
ff = ff.rename(columns={"Mkt-RF": "FF_MKT", "SMB": "FF_SMB", "HML": "FF_HML", "RMW": "FF_RMW", "CMA": "FF_CMA", "RF": "FF_RF"})

data_all = factors.join(ff, how="left").sort_index()
data = data_all.loc[START_P: END_P] if END_P else data_all.loc[START_P:]
data = data.dropna(subset=["PROF"])
F = data * 100   # 以降の回帰は %/月 で表示
print(f"分析に使う月：{data.index.min()}〜{data.index.max()}（{len(data)} か月）、FF5 が揃う月：{data['FF_RMW'].notna().sum()} か月")

min_cnt = pd.DataFrame({s: port_counts[s].min(axis=1) for s in PROFIT_SIGNALS}).loc[data.index.min():]
print("6つのポートフォリオのうち最も銘柄数が少ないものの銘柄数（全期間の最小値）：")
display(min_cnt.min().to_frame("最小銘柄数").T)

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(exist_ok=True)
    data.to_csv(OUTPUT_DIR / "profitability_factors.csv")
    print(f"→ {OUTPUT_DIR / 'profitability_factors.csv'} に保存")

## 9. ファクターの成績

各ファクター（ロング・ショートの月次リターン）の平均・t値（Newey-West）・年率シャープレシオ・最大ドローダウンを比べます。
t値はおおよそ「シャープレシオ × √年数」なので、期間が短い日本では t ≈ 2 を目安にしつつ、指標間の相対比較を重視します。

下のヘルパー関数（`ols_nw`・`reg_table`）は以降の回帰すべてで使います。

In [ ]:
def ols_nw(y, X=None, lags=NW_LAGS):
    """OLS + Newey-West 標準誤差。X=None なら平均の検定"""
    df = y.to_frame("y") if X is None else pd.concat([y.rename("y"), X], axis=1)
    df = df.dropna()
    yv = df["y"].to_numpy()
    names = ["alpha"] + ([] if X is None else list(X.columns))
    Xv = np.column_stack([np.ones(len(df))] + ([] if X is None else [df[c].to_numpy() for c in X.columns]))
    n, k = Xv.shape
    if n < k + 12:   # 標本が足りない
        nan = pd.Series(np.nan, index=names)
        return {"coef": nan, "t": nan, "adj_r2": np.nan, "n": n, "resid_sd": np.nan}
    XtX_inv = np.linalg.pinv(Xv.T @ Xv)
    b = XtX_inv @ Xv.T @ yv
    e = yv - Xv @ b
    u = Xv * e[:, None]
    S = u.T @ u
    for l in range(1, lags + 1):
        G = u[l:].T @ u[:-l]
        S += (1 - l / (lags + 1)) * (G + G.T)
    V = XtX_inv @ S @ XtX_inv * n / max(n - k, 1)
    se = np.sqrt(np.diag(V))
    r2 = 1 - e.var() / yv.var() if yv.var() > 0 else np.nan
    return {
        "coef": pd.Series(b, index=names), "t": pd.Series(b / se, index=names),
        "adj_r2": 1 - (1 - r2) * (n - 1) / max(n - k, 1), "n": n, "resid_sd": e.std(ddof=k),
    }


def reg_table(specs, data_pct):
    """specs: {列名: (y, [x...])} → 「係数 [t値]」の表"""
    cols = {}
    for name, (y, xs) in specs.items():
        r = ols_nw(data_pct[y], data_pct[xs] if xs else None)
        cell = {v: f"{r['coef'][v]:.2f} [{r['t'][v]:.2f}]" for v in r["coef"].index}
        cell["adj.R2(%)"] = f"{100 * r['adj_r2']:.1f}" if xs else ""
        cell["N"] = str(r["n"])
        cols[name] = cell
    out = pd.DataFrame(cols)
    order = ["alpha"] + [i for i in out.index if i not in ("alpha", "adj.R2(%)", "N")] + ["adj.R2(%)", "N"]
    return out.reindex(order).fillna("")


def perf_summary(df):
    rows = {}
    for c in df.columns:
        r = df[c].dropna()
        if len(r) < 12:
            continue
        cum = (1 + r).cumprod()
        res = ols_nw(r * 100)
        rows[c] = {
            "平均(%/月)": 100 * r.mean(), "t値(NW)": res["t"]["alpha"], "年率σ(%)": 100 * r.std() * np.sqrt(12),
            "年率SR": r.mean() / r.std() * np.sqrt(12), "最大DD(%)": 100 * (cum / cum.cummax() - 1).min(),
            "勝率": (r > 0).mean(), "月数": len(r),
        }
    return pd.DataFrame(rows).T


perf_cols = PROFIT_SIGNALS + ["FF_RMW", "HML_L", "FF_HML", "CMA_L", "FF_CMA", "SMB_L", "FF_SMB", "UMD", "MKT_L", "FF_MKT"]
perf = perf_summary(data[[c for c in perf_cols if c in data.columns]])
show(perf, {"平均(%/月)": "{:.3f}", "t値(NW)": "{:.2f}", "年率σ(%)": "{:.2f}", "年率SR": "{:.2f}",
            "最大DD(%)": "{:.1f}", "勝率": "{:.2f}", "月数": "{:.0f}"})

# 自前ファクターと外部ファクターの相関（作り方の確認）
pairs = [("RMW_OP", "FF_RMW"), ("PROF", "FF_RMW"), ("HML_L", "FF_HML"), ("CMA_L", "FF_CMA"), ("SMB_L", "FF_SMB"), ("MKT_L", "FF_MKT")]
print("自前ファクターと外部ファクターの相関（同じものなら高いはず。MKT はドル円の影響で低めになる）")
show(pd.Series({f"{a} vs {b}": data[[a, b]].dropna().corr().iloc[0, 1] for a, b in pairs}).to_frame("相関").T, "{:.2f}")

fig, ax = plt.subplots(figsize=(12, 5))
for c in ["PROF", "RMW_OP", "COP_RD", "GPA", "ROE", "FF_RMW"]:
    r = data[c].dropna()
    ax.plot(r.index.to_timestamp(), (1 + r).cumprod(), label=c, lw=2.2 if c == "PROF" else 1.2)
ax.set_yscale("log")
ax.axhline(1, color="gray", lw=0.8)
ax.set_title("Cumulative return of profitability factors (long-short, log scale)")
ax.legend()
plt.tight_layout()
plt.show()

## 10. 5分位ポートフォリオ（単調性の確認）

ロング・ショートの差だけでなく、指標の低い順に並べた5つのポートフォリオのリターンが**右肩上がり**になっているかを見ます。
分位点は大型株の中で計算し（米国の NYSE 基準に相当）、全銘柄に当てはめます。時価総額加重。
「Q5−Q1」は高収益 − 低収益です。日本では低収益側（Q1）だけが極端に悪い、などの非対称性もここで分かります。

In [ ]:
q_rows, q_t = {}, {}
for s in PROFIT_SIGNALS:
    vw, _ = group_vw(hold, f"{s}_q")
    vw = vw.loc[data.index.min(): data.index.max()]
    vw.columns = [f"Q{int(c)}" for c in vw.columns]
    vw["Q5-Q1"] = vw[f"Q{N_QUANTILES}"] - vw["Q1"]
    q_rows[s] = 100 * vw.mean()
    q_t[s] = ols_nw(100 * vw["Q5-Q1"])["t"]["alpha"]
qtab = pd.DataFrame(q_rows).T
qtab["t(Q5-Q1)"] = pd.Series(q_t)
print("5分位ポートフォリオの平均リターン（%/月、時価総額加重）")
show(qtab, "{:.3f}", cmap="RdYlGn", subset=[f"Q{i}" for i in range(1, N_QUANTILES + 1)], axis=1)

## 11. ファクターモデルに対するアルファ

論文の Fig. 1 / Table A1 と同じく、各収益性ファクターを既存のファクターに回帰したアルファ（%/月）と t 値を見ます。

| モデル | 説明変数 |
|---|---|
| 平均 | なし |
| CAPM | FF_MKT |
| FF3 | FF_MKT, FF_SMB, FF_HML |
| FF3+UMD | 上記 + UMD（論文 Fig. 1 の比較対象） |
| FF5−RMW (+UMD) | FF_MKT, FF_SMB, FF_HML, FF_CMA (+ UMD) |
| FF5 (+UMD) | 上記 + FF_RMW |

収益性ファクターは割安株（HML）と負の相関を持ちやすいので、HML を入れるとアルファが**大きくなる**ことがあります（論文の Novy-Marx 2013 の指摘）。

In [ ]:
MOM = ["UMD"] if INCLUDE_UMD else []
MODELS = {
    "平均": [],
    "CAPM": ["FF_MKT"],
    "FF3": ["FF_MKT", "FF_SMB", "FF_HML"],
    "FF3+UMD": ["FF_MKT", "FF_SMB", "FF_HML", "UMD"],
    "FF5−RMW" + ("+UMD" if MOM else ""): ["FF_MKT", "FF_SMB", "FF_HML", "FF_CMA"] + MOM,
    "FF5" + ("+UMD" if MOM else ""): ["FF_MKT", "FF_SMB", "FF_HML", "FF_CMA", "FF_RMW"] + MOM,
}
Fff = F.dropna(subset=["FF_MKT"])   # 外部CSVがある期間にそろえる
alpha_rows = {}
for s in PROFIT_SIGNALS + ["FF_RMW"]:
    row = {}
    for m, xs in MODELS.items():
        if s in xs:
            row[m] = ""
            continue
        r = ols_nw(Fff[s], Fff[xs] if xs else None)
        row[m] = f"{r['coef']['alpha']:.2f} [{r['t']['alpha']:.2f}]"
    alpha_rows[SIGNAL_LABELS[s]] = row
print(f"アルファ（%/月）[t値]、標本 {Fff.index.min()}〜{Fff.index.max()}")
display(pd.DataFrame(alpha_rows).T)

print("参考：FF3+UMD・FF5 への回帰の係数（PROF と RMW_OP）")
display(reg_table({
    "PROF: FF3+UMD": ("PROF", MODELS["FF3+UMD"]),
    "RMW_OP: FF3+UMD": ("RMW_OP", MODELS["FF3+UMD"]),
    "PROF: FF5": ("PROF", list(MODELS.values())[-1]),
    "RMW_OP: FF5": ("RMW_OP", list(MODELS.values())[-1]),
}, Fff))

## 12. スパニングテスト：PROF は RMW を超えるか（本題）

スパニングテストは「ファクター A に投資できる投資家にとって、ファクター B を追加すると投資機会が広がるか」を調べる回帰です。

$$ \text{PROF} = \alpha + \beta_{\text{RMW}} \text{RMW} + \beta' x + \varepsilon \qquad \text{RMW} = \alpha + \beta_{\text{PROF}} \text{PROF} + \beta' x + \varepsilon $$

- 左の式で **α > 0 が有意** → PROF は RMW と他のファクターでは説明できない（PROF を加えると改善する）。
- 右の式で **α が有意でない** → RMW は PROF で説明し尽くされる（RMW を持つ意味が無い）。

論文（Table A3）では、米国・先進国（日本を含む）・新興国のすべてで「PROF は RMW に対して有意なアルファ、RMW は PROF に対してアルファなし」でした。
これが日本単独でも成り立てば、「測り方を変えると改善する」と言えます。

2種類の RMW で検証します。

- **FF_RMW**（外部CSV）：実務で使われる既存のファクター。ただしドル建て・ユニバースや作り方が違う。
- **RMW_OP**（自前）：PROF と**同じユニバース・同じ作り方**で、R&D の扱いだけ違う。**測り方の効果だけ**を純粋に取り出せる。

説明変数 x は「外部CSVの FF ファクター」版と「自前の円建てファクター」版の両方で回します。

In [ ]:
CONTROL_SETS = {
    "外部FF": ["FF_MKT", "FF_SMB", "FF_HML", "FF_CMA"] + MOM,
    "自前(円建て)": ["MKT_L", "SMB_L", "HML_L", "CMA_L"] + MOM,
}
for cname, x in CONTROL_SETS.items():
    Fx = F.dropna(subset=x)
    specs = {
        "(1) PROF ~ FF_RMW + x": ("PROF", ["FF_RMW"] + x),
        "(2) FF_RMW ~ PROF + x": ("FF_RMW", ["PROF"] + x),
        "(3) PROF ~ RMW_OP + x": ("PROF", ["RMW_OP"] + x),
        "(4) RMW_OP ~ PROF + x": ("RMW_OP", ["PROF"] + x),
        "(5) PROF ~ RMW_OP": ("PROF", ["RMW_OP"]),
        "(6) RMW_OP ~ PROF": ("RMW_OP", ["PROF"]),
    }
    if len(Fx) < 36:
        print(f"■ 説明変数 x = {cname}：標本が足りないため省略（{len(Fx)} か月）")
        continue
    print(f"■ 説明変数 x = {cname}：{x}（標本 {Fx.index.min()}〜{Fx.index.max()}、%/月）")
    display(reg_table(specs, Fx))

## 13. 指標間の総当たりスパニングテスト（論文の Fig. 2 に相当）

論文は「どのクオリティ指標も PROF に対してアルファを持たないが、PROF はどのクオリティ指標に対しても有意なアルファを持つ」ことを示しました。
ここでは、収益性指標のすべての組み合わせについて

$$ y = \alpha + \beta\, x_{\text{指標}} + \beta' x_{\text{コントロール}} + \varepsilon $$

の α の t 値を計算します（コントロールは外部FFの MKT・SMB・HML・CMA と UMD。論文と同じく RMW は除く）。

- **行 y、列 x のセル**：x を持っている投資家が y を加えると改善するか（t値）。
- ある指標の**行が全部大きい**（他のどれでも説明できない）かつ**列が全部小さい**（他を説明し尽くす）なら、その指標が「最良の測り方」。

In [ ]:
MEASURES = PROFIT_SIGNALS + ["FF_RMW"]
ctrl = CONTROL_SETS["外部FF"]
Fm = F.dropna(subset=ctrl + MEASURES)
tmat = pd.DataFrame(np.nan, index=MEASURES, columns=MEASURES)
amat = tmat.copy()
for y in MEASURES:
    for x in MEASURES:
        if x == y:
            continue
        r = ols_nw(Fm[y], Fm[[x] + ctrl])
        tmat.loc[y, x] = r["t"]["alpha"]
        amat.loc[y, x] = r["coef"]["alpha"]
print(f"α の t 値（行 y を 列 x + コントロール に回帰）、標本 {Fm.index.min()}〜{Fm.index.max()}")
show(tmat, "{:.2f}", cmap="RdBu_r", vmin=-4, vmax=4)
print("α（%/月）")
show(amat, "{:.2f}")

# 論文 Fig. 2 と同じ形：PROF と各指標の双方向
fig2 = pd.DataFrame({
    "PROF ~ 指標 + x の α t値": tmat.loc["PROF"].drop("PROF"),
    "指標 ~ PROF + x の α t値": tmat["PROF"].drop("PROF"),
})
ax = fig2.plot.bar(figsize=(11, 4), color=["tab:blue", "tab:red"])
ax.axhline(1.96, color="gray", ls="--", lw=0.8)
ax.axhline(-1.96, color="gray", ls="--", lw=0.8)
ax.set_title("Spanning tests: PROF vs. other measures (t-stat of alpha)")
ax.legend(["PROF on measure + controls", "measure on PROF + controls"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 14. 最大シャープレシオの比較

スパニングテストを投資の言葉に言い換えると、「ファクターの組み合わせで達成できる最大シャープレシオが上がるか」です。
ファクター集合 f の最大二乗シャープレシオは $SR^2 = \mu' \Sigma^{-1} \mu$ で、
ファクター B を加えたときの増分は B のアルファの情報レシオの2乗 $(\alpha / \sigma_\varepsilon)^2$ に等しくなります。

- **FF5 の RMW を PROF に入れ替える**とシャープレシオが上がるか
- **FF5 に PROF を加える**とどれだけ上がるか

を比べます（標本は全集合で共通。事後的な最大値なので水準は上振れしますが、集合間の比較には使えます）。

In [ ]:
def max_sr(df):
    mu = df.mean().to_numpy()
    S = df.cov().to_numpy()
    sh2 = float(mu @ np.linalg.solve(S, mu))
    return np.sqrt(12 * sh2)


core = ["FF_MKT", "FF_SMB", "FF_HML", "FF_CMA"]
sets = {
    "FF4（収益性なし）": core,
    "FF5（外部RMW）": core + ["FF_RMW"],
    "FF4 + RMW_OP（自前・R&D控除）": core + ["RMW_OP"],
    "FF4 + PROF（論文）": core + ["PROF"],
    "FF5 + PROF": core + ["FF_RMW", "PROF"],
    "FF4 + RMW_OP + PROF": core + ["RMW_OP", "PROF"],
    "FF4 + COP_RD": core + ["COP_RD"],
    "FF4 + GPA": core + ["GPA"],
    "FF4 + ROE": core + ["ROE"],
}
if INCLUDE_UMD:
    sets.update({f"{k} + UMD": v + ["UMD"] for k, v in list(sets.items())})
all_cols = sorted({c for v in sets.values() for c in v})
Fs = data[all_cols].dropna()
srtab = pd.DataFrame({k: {"最大シャープレシオ(年率)": max_sr(Fs[v]), "ファクター数": len(v)} for k, v in sets.items()}).T
print(f"標本 {Fs.index.min()}〜{Fs.index.max()}（{len(Fs)} か月）")
show(srtab, {"最大シャープレシオ(年率)": "{:.3f}", "ファクター数": "{:.0f}"})

## 15. Fama-MacBeth 回帰（論文の Table B3 に相当）

ポートフォリオではなく個別銘柄の断面回帰で、どの収益性指標が翌月のリターンを最もよく予測するかを比べます。論文と同じく

- 毎月、個別銘柄の月次リターンを収益性指標とコントロール（総資産成長率 ln(AT/AT₋₁)、ln(BM)、ln(時価総額)、過去12〜2か月リターン、前月リターン）に回帰
- 時価総額で加重した最小二乗法（WLS）で、小型株の影響を抑える
- 説明変数は毎月 1%・99% 点の外側を切り落とす
- 係数の時系列平均と t 値を報告（係数は %/月）

論文では **OP_RD（= PROF の指標）が OP（FF 型）とキャッシュベースの COP を両方とも包含**しました（列 (5)(6) で OP・COP が有意でなくなる）。
日本でも同じことが起きるかを見ます。

In [ ]:
fm = hold.loc[(hold["month"] >= data.index.min()) & (hold["month"] <= data.index.max())].copy()
fm = fm[fm["me_lag"] > 0]
fm["ret_pct"] = 100 * fm["ret"]
fm["ln_AG"] = np.log1p(fm["INV"].where(fm["INV"] > -1))
fm["ln_BM"] = np.log(fm["BM"].where(fm["BM"] > 0))
fm["ln_ME"] = np.log(fm["me_lag"])
fm = fm.rename(columns={"PROF": "OP_RD", "RMW_OP": "OP", "COP_RD": "COP_RD"})
FMB_CONTROLS = ["ln_AG", "ln_BM", "ln_ME", "r12_2", "r1_0"]


fm = fm.sort_values("month").reset_index(drop=True)
FMB_VARS = ["OP", "OP_RD", "COP", "COP_RD", "GPA", "ROE"] + FMB_CONTROLS
for c in FMB_VARS:   # 毎月 1%・99% 点の外側を切り落とすためのフラグ（変数ごとに事前計算）
    gq = fm.groupby("month")[c]
    fm[f"_in_{c}"] = fm[c].between(gq.transform("quantile", FMB_TRIM[0]), gq.transform("quantile", FMB_TRIM[1]))
_month_codes, _month_start = np.unique(fm["month"].astype(str).to_numpy(), return_index=True)
_bounds = list(zip(_month_start, list(_month_start[1:]) + [len(fm)]))


def fama_macbeth(df, xs, weight="me_lag"):
    """月次の WLS 断面回帰 → 係数の時系列平均と t 値"""
    Y = df["ret_pct"].to_numpy()
    W = df[weight].to_numpy()
    Xall = df[xs].to_numpy()
    ok_all = np.isfinite(Y) & np.isfinite(W) & np.isfinite(Xall).all(axis=1) & df[[f"_in_{c}" for c in xs]].all(axis=1).to_numpy()
    coefs, r2s, ns, months_used = [], [], [], []
    for (s, e), mlabel in zip(_bounds, _month_codes):
        ok = ok_all[s:e]
        n = int(ok.sum())
        if n < max(FMB_MIN_OBS, len(xs) + 5):
            continue
        sw = np.sqrt(W[s:e][ok])
        X = np.column_stack([np.ones(n), Xall[s:e][ok]]) * sw[:, None]
        y = Y[s:e][ok] * sw
        b, *_ = np.linalg.lstsq(X, y, rcond=None)
        res = y - X @ b
        r2 = 1 - (res @ res) / (((y - y.mean()) ** 2).sum())
        coefs.append(b)
        r2s.append(1 - (1 - r2) * (n - 1) / (n - len(xs) - 1))
        ns.append(n)
        months_used.append(mlabel)
    C = pd.DataFrame(coefs, columns=["const"] + xs, index=months_used)
    t = {c: ols_nw(C[c])["t"]["alpha"] for c in C.columns}
    cell = {c: f"{C[c].mean():.2f} [{t[c]:.2f}]" for c in C.columns if c != "const"}
    cell.update({"adj.R2(%)": f"{100 * np.mean(r2s):.1f}", "平均銘柄数": f"{np.mean(ns):,.0f}", "月数": str(len(C))})
    return cell


fmb_specs = {
    "(1)": ["OP"], "(2)": ["OP_RD"], "(3)": ["COP"], "(4)": ["COP_RD"],
    "(5)": ["OP", "OP_RD"], "(6)": ["COP", "OP_RD"], "(7)": ["OP_RD", "COP_RD"],
    "(8)": ["GPA"], "(9)": ["ROE"], "(10)": ["OP_RD", "GPA"], "(11)": ["OP_RD", "ROE"],
}
fmb = pd.DataFrame({k: fama_macbeth(fm, v + FMB_CONTROLS) for k, v in fmb_specs.items()})
order = ["OP", "OP_RD", "COP", "COP_RD", "GPA", "ROE"] + FMB_CONTROLS + ["adj.R2(%)", "平均銘柄数", "月数"]
print("Fama-MacBeth 回帰（WLS・時価総額加重）係数 %/月 [t値]")
display(fmb.reindex(order).fillna(""))

## 16. 期間別の頑健性

日本の収益性ファクターの評価は期間によって大きく変わることが知られています。`SUBPERIODS` の各期間について、
平均リターンと、PROF vs RMW のスパニングテストのアルファを確認します。あわせて60か月移動平均リターンの推移を描きます。

In [ ]:
sub_rows = {}
for s0, s1 in SUBPERIODS:
    p0 = pd.Period(s0, freq="M") if s0 else data.index.min()
    p1 = pd.Period(s1, freq="M") if s1 else data.index.max()
    Fp = F.loc[max(p0, data.index.min()): p1]
    if len(Fp) < 24:
        continue
    x = CONTROL_SETS["外部FF"]
    Fpx = Fp.dropna(subset=x + ["FF_RMW"])
    row = {}
    for c in ["PROF", "RMW_OP", "COP_RD", "GPA", "ROE", "FF_RMW"]:
        r = ols_nw(Fp[c])
        row[f"{c} 平均"] = f"{r['coef']['alpha']:.2f} [{r['t']['alpha']:.2f}]"
    for lab, (y, xs) in {
        "PROF|FF5(+UMD) α": ("PROF", x + ["FF_RMW"]),
        "PROF|RMW_OP+x α": ("PROF", ["RMW_OP"] + x),
        "RMW_OP|PROF+x α": ("RMW_OP", ["PROF"] + x),
        "FF_RMW|PROF+x α": ("FF_RMW", ["PROF"] + x),
    }.items():
        r = ols_nw(Fpx[y], Fpx[xs])
        row[lab] = f"{r['coef']['alpha']:.2f} [{r['t']['alpha']:.2f}]"
    row["月数"] = str(len(Fp))
    sub_rows[f"{Fp.index.min()}〜{Fp.index.max()}"] = row
print("期間別（%/月 [t値]）")
display(pd.DataFrame(sub_rows))

fig, ax = plt.subplots(figsize=(12, 4))
for c in ["PROF", "RMW_OP", "FF_RMW", "GPA"]:
    roll = (100 * data[c]).rolling(60, min_periods=48).mean()
    ax.plot(roll.index.to_timestamp(), roll, label=c, lw=2.2 if c == "PROF" else 1.2)
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("60-month rolling mean return (%/month)")
ax.legend()
plt.tight_layout()
plt.show()

## 17. 結果の読み方

「収益性の測り方を変えたら日本でも効くようになったか」は、次の順に判断できます。

| 見る場所 | 改善したと言える結果 |
|---|---|
| 7. 診断 | PROF と RMW_OP の順位相関が十分に1より小さく、区分が変わる銘柄がある程度ある（**違いが無ければ以下の結果も同じになるのは当然**） |
| 9. 成績 | PROF の平均・t値・シャープレシオが RMW_OP・FF_RMW より高い |
| 10. 5分位 | PROF の Q1→Q5 がより右肩上がり |
| 11. アルファ | PROF の FF5(+UMD) アルファが正で有意（既存の RMW を入れても残る） |
| 12. スパニング | (3) PROF ~ RMW_OP の α が有意に正、かつ (4) RMW_OP ~ PROF の α が有意でない（**測り方の純粋な効果**） |
| 13. 総当たり | PROF の行が大きく、PROF の列が小さい |
| 14. 最大SR | 「FF4 + PROF」>「FF5（外部RMW）」、「FF4 + RMW_OP + PROF」>「FF4 + RMW_OP」 |
| 15. FMB | OP_RD が OP・COP を包含（列 (5)(6) で OP・COP が有意でなくなる） |
| 16. 期間別 | 上の結果が特定の期間だけに依存していない |

**注意点**

- **通貨**：外部CSV（Ken French の Japan ファクター）はドル建て。ロング・ショートのファクターは通貨の影響がほぼ打ち消されますが、市場ファクターは為替を含みます。そのため自前の円建てファクターでも検証しています。
- **円金利**：`LOCAL_RF_MONTHLY` は既定で0です。円の短期金利があれば設定してください（ロング・ショートの比較にはほぼ影響しません）。
- **R&D の範囲**：日本基準では R&D の一部が売上原価に入るため、注記の研究開発費総額を足し戻すと論文よりやや広い調整になります。また IFRS 採用企業は開発費の資産計上があり、営業利益の定義も日本基準と異なります（`CTYPE_PRIORITY` で日本基準を優先しています）。
- **アクルーアル**：前払費用・前受収益・未払費用を取得していないので、Ball et al. (2016) の簡略版です。
- **BE+MIB**：2006年より前の `EQTY`（資本合計）は少数株主持分を含みません。
- **上場廃止**：上場廃止月のリターンがデータに無い場合、その銘柄は廃止月以降ポートフォリオから外れます。
- **証券コード**：保有期間中は組成時点の証券コードで追跡します（期中のコード変更は追いません）。
- **多重検定**：指標を多く比べるほど、偶然どれかが良く見える可能性が高まります。論文の結論（PROF が最良）を**事前に決めた1つの仮説**として検証するのが筋です。